In [3]:
import numpy as np
import h5py
from pathlib import Path

# ============================================================
# 1. Parameters
# ============================================================

root = Path(
    "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis"
)

FF_path = root / (
    "FF_symmetric_q0_0_0_hyp0-1-2-3-4-5-6-7-8_fb_tins18_ncfg799.h5"
)

pt2_path = root / (
    "799cfgs/"
    "2pt_N40_rho3.25_G5_ez_momfrac0p6_fb_tsep18_ncfg799.h5"
)

# Array indices
TGF = 3
W = 1
PF = 5
TSEP = 2
TAU = 6

# "real" or "imag"
PART = "real"


# ============================================================
# 2. Jackknife utilities
# ============================================================

def jackknife(x):
    n = x.shape[0]
    return (np.sum(x, axis=0) - x) / (n - 1)


def jk_error(x):
    n = len(x)
    mean = np.mean(x)
    return np.sqrt(
        (n - 1) / n * np.sum(np.abs(x - mean)**2)
    )


# ============================================================
# 3. Load original lattice data (READ ONLY)
# ============================================================

with h5py.File(FF_path, "r") as f, \
     h5py.File(pt2_path, "r") as p:

    # Verify configuration alignment if both lists exist
    if "cfg_list" in f and "cfg_list" in p:
        assert np.array_equal(
            f["cfg_list"][:],
            p["cfg_list"][:]
        ), "Configuration lists are not aligned!"

    ff = f["FF_forward"]

    # O3 = TXTX + TYTY - 2 XYXY
    # Shape: [cfg, tsrc]
    G = (
        ff[:, :, 3, 3, TGF, W, 0, TAU]
        + ff[:, :, 4, 4, TGF, W, 0, TAU]
        - 2 * ff[:, :, 0, 0, TGF, W, 0, TAU]
    )

    # Nucleon 2pt
    # Original shape:
    # [cfg, tsrc, xsrc, ysrc, zsrc, pf, tsep]

    B = p["pt2_forward"][
        :, :, :, :, :, PF, TSEP
    ]

    # Average spatial sources
    # Shape: [cfg, tsrc]
    B = B.mean(axis=(2, 3, 4))


assert B.shape == G.shape

Ncfg, Nsrc = B.shape

print(f"Ncfg = {Ncfg}")
print(f"Nsrc = {Nsrc}")
print(f"B shape = {B.shape}")
print(f"G shape = {G.shape}")


# ============================================================
# 4. Original vacuum-subtracted 3pt (Method 1)
# ============================================================

# Average over tsrc within each configuration

BG_cfg = np.mean(B * G, axis=1)
B_cfg = np.mean(B, axis=1)
G_cfg = np.mean(G, axis=1)

# Original full-sample estimator
Y_full = (
    np.mean(BG_cfg)
    - np.mean(B_cfg) * np.mean(G_cfg)
)

# Jackknife
Y_jk_complex = (
    jackknife(BG_cfg)
    - jackknife(B_cfg) * jackknife(G_cfg)
)

# Select real or imaginary part
if PART == "real":
    Y_jk = Y_jk_complex.real
else:
    Y_jk = Y_jk_complex.imag

error_original = jk_error(Y_jk)


# ============================================================
# 5. Your proposed auxiliary variable
#
# Z_i = FF_i - <FF>
#
# Here <FF> is averaged over configurations
# separately for each tsrc.
# ============================================================

# [1, tsrc]
G_mean_cfg = np.mean(G, axis=0, keepdims=True)

# [cfg, tsrc]
Z = G - G_mean_cfg

# Average over tsrc
# [cfg]
Z_cfg = np.mean(Z, axis=1)

# Jackknife, keeping the full-sample FF mean fixed
Z_jk_complex = jackknife(Z_cfg)

if PART == "real":
    Z_jk = Z_jk_complex.real
else:
    Z_jk = Z_jk_complex.imag

print("\nAuxiliary variable checks:")
print("Full sample mean Z =", np.mean(Z_cfg))
print("JK error of Z =", jk_error(Z_jk))


# ============================================================
# 6. Calculate optimal alpha
# ============================================================

Y_centered = Y_jk - np.mean(Y_jk)
Z_centered = Z_jk - np.mean(Z_jk)

cov_YZ = np.sum(Y_centered * Z_centered)
var_Z = np.sum(Z_centered**2)

if var_Z == 0:
    raise ValueError("Auxiliary variable has zero variance.")

alpha_opt = cov_YZ / var_Z

rho = cov_YZ / np.sqrt(
    np.sum(Y_centered**2) * var_Z
)

print("\nControl variate:")
print(f"alpha_opt = {alpha_opt:.8e}")
print(f"Correlation coefficient = {rho:.6f}")


# ============================================================
# 7. Your proposed improved JK estimator
# ============================================================

Y_jk_proposed = Y_jk - alpha_opt * Z_jk

error_proposed = jk_error(Y_jk_proposed)


# ============================================================
# 8. Correct JK: recompute <FF> in each JK subset
# ============================================================

Z_jk_correct = np.zeros(Ncfg, dtype=float)

for i in range(Ncfg):

    # Delete configuration i
    G_subset = np.delete(G, i, axis=0)

    # Recalculate <FF> using ONLY the remaining cfgs
    G_subset_mean = np.mean(
        G_subset, axis=0, keepdims=True
    )

    # Construct centered FF again
    Z_subset = G_subset - G_subset_mean

    # Average over cfg and tsrc
    z_value = np.mean(Z_subset)

    if PART == "real":
        Z_jk_correct[i] = z_value.real
    else:
        Z_jk_correct[i] = z_value.imag

Y_jk_correct = (
    Y_jk - alpha_opt * Z_jk_correct
)

error_correct = jk_error(Y_jk_correct)


# ============================================================
# 9. Compare results
# ============================================================

print("\n" + "=" * 60)
print("CONTROL VARIATE TEST RESULTS")
print("=" * 60)

print(f"Operator: O3 = TXTX + TYTY - 2 XYXY")
print(f"q = 0")
print(f"PF index = {PF}")
print(f"TGF index = {TGF}")
print(f"Wilson-line index = {W}")
print(f"TSEP = {TSEP}, TAU = {TAU}")
print(f"Component = {PART}")

print("\nOriginal full-sample 3pt:")
print(Y_full)

print("\nJackknife errors:")
print(f"Original       : {error_original:.8e}")
print(f"Your method    : {error_proposed:.8e}")
print(f"Recomputed JK  : {error_correct:.8e}")

print("\nError ratios:")
print(
    f"Your / Original = "
    f"{error_proposed/error_original:.6f}"
)
print(
    f"Correct / Original = "
    f"{error_correct/error_original:.6f}"
)

print("\nVariance reduction factors:")
print(
    f"Apparent = "
    f"{(error_original/error_proposed)**2:.4f}"
)
print(
    f"Recomputed = "
    f"{(error_original/error_correct)**2:.4f}"
)

print("\nConsistency checks:")
print(
    "Full-sample correction =",
    np.mean(Z_cfg)
)
print(
    "Max recomputed JK correction =",
    np.max(np.abs(Z_jk_correct))
)
print(
    "Max difference from original JK =",
    np.max(np.abs(Y_jk_correct - Y_jk))
)

print("=" * 60)

Ncfg = 799
Nsrc = 8
B shape = (799, 8)
G shape = (799, 8)

Auxiliary variable checks:
Full sample mean Z = (4.179663151530001e-16+9.379447558948201e-33j)
JK error of Z = 0.41678618844954857

Control variate:
alpha_opt = -4.14885874e-08
Correlation coefficient = -0.047031

CONTROL VARIATE TEST RESULTS
Operator: O3 = TXTX + TYTY - 2 XYXY
q = 0
PF index = 5
TGF index = 3
Wilson-line index = 1
TSEP = 2, TAU = 6
Component = real

Original full-sample 3pt:
(-3.759668435239589e-07-1.673057324193069e-07j)

Jackknife errors:
Original       : 3.67672936e-07
Your method    : 3.67266087e-07
Recomputed JK  : 3.67672936e-07

Error ratios:
Your / Original = 0.998893
Correct / Original = 1.000000

Variance reduction factors:
Apparent = 1.0022
Recomputed = 1.0000

Consistency checks:
Full-sample correction = (4.179663151530001e-16+9.379447558948201e-33j)
Max recomputed JK correction = 1.2109500258568123e-15
Max difference from original JK = 5.293955920339377e-23
